# <a id='toc1_'></a>[beancount-jupyter notebook for exploaring a beancount file](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [beancount-jupyter notebook for exploaring a beancount file](#toc1_)    
  - [Notebook preparations](#toc1_1_)    
  - [Checks](#toc1_2_)    
    - [Beancount built in checks](#toc1_2_1_)    
  - [Misc](#toc1_3_)    
  - [Chart of accounts](#toc1_4_)    
  - [Net Worth over time](#toc1_5_)    
    - [In original currencies](#toc1_5_1_)    
    - [Converted to operating currency](#toc1_5_2_)    
  - [Financial overview](#toc1_6_)    
  - [Exploring expenses](#toc1_7_)    
  - [Filtering spesific transactions](#toc1_8_)    
  - [Prices information](#toc1_9_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Notebook preparations](#toc0_)

In [1]:
#@title Importing modules

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Run shell commands only in Colab
if IN_COLAB:
    %pip install beanquery
    %pip install git+https://github.com/Ev2geny/evbeantools.git@develop_pr
    %pip install requests

    print("Packages installed in Colab environment")
else:
    print("Running locally, skipping pip install")

import os
import re
from io import StringIO
import datetime
from collections.abc import Iterable
from pprint import pprint
from decimal import Decimal

import ipywidgets as widgets
from IPython.display import display

import pandas as pd
import numpy as np
from pandas.io.formats.style import Styler

import requests

import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import plotly.graph_objects as go
import plotly.io as pio

from schema import Schema, Optional, Or, SchemaError

from beancount.loader import load_file, load_string
from beancount.parser import printer
from beancount.parser.printer import print_entries

from beancount.core.data import Transaction, Posting, Open, Close, Balance, Price, Note, Event, Query, Custom
from beancount.core import prices

from evbeantools.sing_curr_conv import get_equiv_sing_curr_entries
from evbeantools.juptools import  add_total, beanquery2df,  get_net_worths, get_bean_pivot, get_period_end_dates, highlight_rows, get_net_worths_per_commodity
from evbeantools.juptools import get_sunburst_figure_from_pivot, show_interactive_fin_flow_diag
from evbeantools.beanfuncs import check_mult_funds_in_transit, split_posting
from evbeantools.printer_rich import display_entries

if IN_COLAB:
    import requests

Running locally, skipping pip install


<style>
h1 { color: purple; }
h2 { color: orange; }
</style>

In [2]:
pd.set_option('display.float_format', '{:.2f}'.format)
pd.options.display.precision=2

FILE_URL = "https://raw.githubusercontent.com/beancount/fava/main/contrib/examples/example.beancount"

BEAN_FILE_NAME = "example-fava.beancount"

IN_COLAB = True

if IN_COLAB:
    response = requests.get(FILE_URL)

    # Check if the request was successful
    if response.status_code == 200:
        entries, errors, options = load_string(response.text)
    else:
        print(f"Failed to download file. Status code: {response.status_code}")

else:
    entries, errors, options = load_file(BEAN_FILE_NAME)

CURR = options["operating_currency"][0]

## <a id='toc1_2_'></a>[Checks](#toc0_)

### <a id='toc1_2_1_'></a>[Beancount built in checks](#toc0_)

In [3]:
printer.print_errors(errors)

## <a id='toc1_3_'></a>[Misc](#toc0_)

In [4]:
op_currs = options["operating_currency"]
op_currs

['USD']

## <a id='toc1_4_'></a>[Chart of accounts](#toc0_)

In [5]:
#getting Chart of Accounts

def get_chart_of_accounts(entries, opts) -> pd.DataFrame:

    query = """
    SELECT account, open.currencies as currencies, open.date as open_date, close.date as close_date
    FROM #accounts
    ORDER BY account
    """

    coa_df = beanquery2df(entries, opts, query).fillna("")
    coa_df.set_index("account", inplace=True)

    coa_df_styled = coa_df.style.set_table_styles([
        {'selector': 'th.row_heading',  # Targets the index column
        'props': [('text-align', 'left')]}
    ])

    return coa_df_styled

get_chart_of_accounts(entries, options)

,currencies,open_date,close_date
account,,,
Assets:US:BofA,,2015-01-01,
Assets:US:BofA:Checking,['USD'],2015-01-01,
Assets:US:ETrade:Cash,['USD'],2015-01-01,
Assets:US:ETrade:GLD,['GLD'],2015-01-01,
Assets:US:ETrade:ITOT,['ITOT'],2015-01-01,
Assets:US:ETrade:VEA,['VEA'],2015-01-01,
Assets:US:ETrade:VHT,['VHT'],2015-01-01,
Assets:US:Federal:PreTax401k,['IRAUSD'],1980-05-12,
Assets:US:Hoogle:Vacation,['VACHR'],2015-01-01,


## <a id='toc1_5_'></a>[Net Worth over time](#toc0_)

### <a id='toc1_5_1_'></a>[In original currencies](#toc0_)

In [6]:
net_worths_per_comm = get_net_worths_per_commodity(entries, options, "Q")
net_worths_per_comm

date,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,2017Q3
commodity,,,,,,,,,,,
GLD,0.00,0.00,5.00,10.00,10.00,12.00,17.00,17.00,9.00,9.00,17.00
IRAUSD,9600.00,2400.00,0.00,0.00,10800.00,2400.00,0.00,0.00,11300.00,2900.00,0.00
ITOT,0.00,0.00,18.00,18.00,18.00,26.00,45.00,95.00,104.00,104.00,104.00
RGAGX,86.72,168.71,196.82,196.82,278.36,356.24,394.75,394.75,474.63,553.75,597.75
USD,3350.24,1420.66,2600.63,7235.17,5199.69,6073.34,154.34,4994.63,2738.81,4107.23,906.58
VACHR,35.00,65.00,28.00,63.00,93.00,64.00,94.00,25.00,55.00,-38.00,-13.00
VBMPX,36.02,65.36,74.96,74.96,100.39,124.73,136.63,136.63,160.04,181.59,193.44
VEA,0.00,0.00,8.00,16.00,16.00,19.00,36.00,36.00,40.00,40.00,36.00
VHT,0.00,0.00,9.00,27.00,27.00,13.00,39.00,39.00,45.00,45.00,64.00


### <a id='toc1_5_2_'></a>[Converted to operating currency](#toc0_)

In [7]:
opp_curr_selector = widgets.Dropdown(
    options=op_currs,
    value=op_currs[0],
    description='Operating Currency:',
    style = {'description_width': 'initial'},
    disabled=False,
)

display(opp_curr_selector)
OP_CURR = opp_curr_selector.value

Dropdown(description='Operating Currency:', options=('USD',), style=DescriptionStyle(description_width='initia…

In [8]:
net_worths_per_comm_conv = get_net_worths_per_commodity(entries, options, "Q", currency=OP_CURR)
net_worths_per_comm_conv

date,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,2017Q3
commodity,,,,,,,,,,,
IRAUSD,9600.00,2400.00,0.00,0.00,10800.00,2400.00,0.00,0.00,11300.00,2900.00,0.00
USD,15784.78,24112.68,33848.71,43483.51,53552.15,67404.66,73460.84,82450.23,93279.05,111164.11,117649.49
VACHR,35.00,65.00,28.00,63.00,93.00,64.00,94.00,25.00,55.00,-38.00,-13.00


In [9]:
def get_plotly_fig_all_axes_right(df: pd.DataFrame,
                                  gap: float = 0.05) -> go.Figure:
    """
    One Y-axis per commodity, all on the right, each clearly separated.
    Works no matter how many rows/columns the DataFrame has.
    """
    # ── 1 ▸ tidy wide → long ─────────────────────────────────────────────
    df = df.copy()
    df.columns = df.columns.astype(str)
    df.index.name  = "commodity"
    df.columns.name = "period"

    long = (df.reset_index()
              .melt(id_vars="commodity",
                    var_name="period",
                    value_name="value"))
    long["period"] = pd.Categorical(long["period"],
                                    categories=df.columns,
                                    ordered=True)

    n_axes = len(df)
    gap    = float(gap)

    domain_end = max(0.1, 1.0 - gap * n_axes)          # keep some plot area
    if domain_end < 1.0 - gap * n_axes:                # squeezed?  recompute
        gap = (1.0 - domain_end) / n_axes

    # ── 2 ▸ base figure (shrunken x-domain leaves room for axes) ─────────
    fig = go.Figure()
    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        title="Net worth at period ends per commodity (separate scales)",
        xaxis=dict(domain=[0, domain_end],
                   type="category",
                   title="Period end date")
    )

    # ── 3 ▸ add trace + axis for each commodity ──────────────────────────
    for i, (commodity, grp) in enumerate(long.groupby("commodity", sort=False)):
        axis_suffix = "" if i == 0 else str(i + 1)   # '', '2', '3', …
        axis_name   = f"yaxis{axis_suffix}"
        yref        = f"y{axis_suffix}"

        # position: 1.0, 1.0-gap, 1.0-2·gap, …
        position = 1.0 - i * gap

        # trace
        fig.add_trace(
            go.Scatter(
                x=grp["period"],
                y=grp["value"],
                mode="lines+markers",
                name=commodity,
                yaxis=yref
            )
        )

        # axis definition
        axis_cfg = dict(
            side="right",
            anchor="free",
            position=position,
            showgrid=False,
            zeroline=False,
            title=commodity
        )
        # only *extra* axes overlay the primary one
        if i > 0:
            axis_cfg["overlaying"] = "y"

        fig.update_layout(**{axis_name: axis_cfg})

    return fig


In [10]:
fig = get_plotly_fig_all_axes_right(net_worths_per_comm_conv)
fig.show()

In [11]:
analys_curr_selector = widgets.Dropdown(
    options=net_worths_per_comm_conv.index.tolist(),
    value=OP_CURR,
    description='Currency for further Analysis:',
    style = {'description_width': 'initial'},
    disabled=False,
)
display(analys_curr_selector)


Dropdown(description='Currency for further Analysis:', index=1, options=('IRAUSD', 'USD', 'VACHR'), style=Desc…

In [12]:
ANALYSIS_CURR = analys_curr_selector.value
ANALYSIS_CURR

'USD'

## <a id='toc1_6_'></a>[Financial overview](#toc0_)

Financial overview combines information about the changes of the **Total Net Worth** over time as well as the factors, which caused these changes. I am not sure there is a formal name to this report in the world of the professional finance, but I find such report very informative for the purposes of understanding personal finance. 

It is inspired by the report of the [Gainstrack](https://www.reddit.com/r/plaintextaccounting/comments/19c1xv7/)

To be able to explain the change in the **Net Worth** between any 2 periods we in general case need to know information about unrealized gains, which are not explicitly available in the beancount ledger. To derive this information we need to convert original entries to the equivalent singe currency entries by using the **get_equiv_sing_curr_entries** function from the [evbeantools.sing_curr_conv](https://github.com/Ev2geny/evbeantools/blob/main/docs/sing_curr_conv.md) package. The equivalent single currency entries will include unrealized gains transactions for every price change. The unrealyzed gains are by default collected in the accounts with the naming convention **Income:Unrealized-Gains:BBB-AAA**, where **BBB** is the target / oparational currency and the **AAA** is the currency, which price change in relation to the currency BBB has caused an unrealized gain. 

In [13]:
entries_eq, errors_eq, opts_eq = get_equiv_sing_curr_entries(entries, options, target_currency='USD', self_testing_mode=True)

In [14]:
#@title get_fin_overview_info function

def get_fin_overview_table(entries,
                           opts,
                           freq,
                           foc:dict,
                           *,
                           start_period = None,
                           qnt_periods = None,
                           end_period = None,
                           currency = None) -> Styler:
    """
    Function to get the financial overview table for the period from start_period to end_period

    params:
        entries: list of entries
        opts: dict of options
        freq: str
            pandas period alias (Y, Q, M, W, D)
                https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-period-aliases
                
        foc: financial overview configuration dictionary

        start_period: Period, str, datetime, date or pandas.Timestamp  . pandas timeperiod representation
                    If not provided, the first date of the first entry will be used to determine the start_period

        qnt_periods: int
            number of periods to display

        end_period: Period, str, datetime, date or pandas.Timestamp  . pandas timeperiod representation
                    Used only if qnt_periods is not provided.
                    If not provided, the last date of the last entry will be used to determine the end_period

    """
    if not start_period:
        start_period = pd.Period(entries[0].date, freq)
    else:
        start_period = pd.Period(start_period, freq)

    if qnt_periods:
        end_period = start_period + qnt_periods - 1
    else:
        if not end_period:
            end_period = pd.Period(entries[-1].date, freq)
        else:
            end_period = pd.Period(end_period, freq)

    if not currency:
        currency = opts["operating_currency"][0]

    # This is just to show types
    start_period: pd.Period = start_period
    end_period: pd.Period = end_period

    start_date_iso_str = start_period.start_time.strftime("%Y-%m-%d")
    end_date_iso_str = end_period.end_time.strftime("%Y-%m-%d")

    def get_table_row(query, name):
        """Performs standard actions to create a table row for Expenses, Income, and Equity accounts

        Returns:
            _type_: _description_
        """
        df = beanquery2df(entries, opts,  query)
        
        # print(f"name: {name}")
        
        # print(df)
        
        # Adding the 'period' column
        df['period'] = pd.PeriodIndex(df['date'], freq=freq)
        
        # Creating the pivot table, where the period is in columns
        
        values_col_name = f'amount ({currency})'
        
        # Handling the case when there are no values in the pivot table for this row
        # In this case, we create an empty empty row dataframe 
        if values_col_name in df.columns:
        
            df_pivot = df.pivot_table(values=values_col_name,
                                    aggfunc='sum', columns='period')
            df_pivot.index = [name]
            
        else:
            df_pivot = pd.DataFrame(index=[name])
            
        return df_pivot

    query_exp=f"""
            select id, date, CONVERT(position,'{currency}', date) as amount, narration
            WHERE
               ({foc["expenses"]["sql"]}) AND
                date >= {start_date_iso_str} AND
                date <= {end_date_iso_str}
            """
    df_expen_pivot_line = get_table_row(query_exp, foc["expenses"]["name"])

    # print(df_expen_pivot_line)


    query_income = f"""
            select id, date, CONVERT(position,'{currency}',date) as amount, narration
            WHERE
                ({foc["income"]["sql"]})  AND
                date >= {start_date_iso_str} AND
                date <= {end_date_iso_str}
            """

    df_income_pivot_line = get_table_row(query_income, foc["income"]["name"])

    # print(df_income_pivot_line)

    query_gains = f"""
            select id, date, CONVERT(position,'{currency}',date) as amount, narration
            WHERE
                ({foc["gains"]["sql"]}) AND
                date >= {start_date_iso_str} AND
                date <= {end_date_iso_str}
            """

    df_gains_pivot_line = get_table_row(query_gains, foc["gains"]["name"])


    query_equity=f"""
            select id, date, CONVERT(position,'{currency}',date) as amount, narration
            WHERE
               ({foc["equity"]["sql"]}) AND
                date >= {start_date_iso_str} AND
                date <= {end_date_iso_str}
            """
    df_equity_pivot_line = get_table_row(query_equity, foc["equity"]["name"])

    # print(df_equity_pivot_line)


    df_result = pd.concat([df_expen_pivot_line, df_income_pivot_line, df_gains_pivot_line, df_equity_pivot_line])

    df_result = add_total(df_result, col_total_name=foc["total_nw_change"])

    # for the purposes of NW calculation we need to calculate NW also for the start_period -1
    net_worth_dates: list[datetime.date] = get_period_end_dates(start_period -1 , end_period)




    neth_worths:pd.DataFrame = get_net_worths(entries, opts, net_worth_dates, currency, num_acc_components_from_root=1, repeat_row_labels=False)

    # print("---------neth_worths -------------")
    # print(neth_worths)
    # neth_worths.columns = neth_worths.columns.droplevel(0)

    # This line converts
    neth_worths = neth_worths.loc[:,f"amount ({currency})"]

    # print('-------- neth_worths = neth_worths.loc[:,f"amount ({currency})"] -------------')
    # print(neth_worths)

    # neth_worths.set_index('acc_L0', inplace=True)
    neth_worths = add_total(neth_worths, col_total_name=foc["total_nw"])

    neth_worths.columns = pd.to_datetime(neth_worths.columns).to_period(freq)


    # print(neth_worths)

    # Adding the net worths to the result
    df_result = pd.concat([df_result, neth_worths], axis=0)

    # print(df_result)

    df_result = df_result.reindex(sorted(df_result.columns), axis=1)

    df_result.fillna(0, inplace=True)


    df_result.loc[foc["unexplained_diff"]] = float('nan')

    df_result.loc[foc["unexplained_diff"], df_result.columns[1:]] = (
        df_result.loc[foc["total_nw"],
        df_result.columns[1:]] - df_result.loc[foc["total_nw"],
        df_result.columns[:-1]].values + df_result.loc[foc["total_nw_change"],
        df_result.columns[1:]].values
    )

    df_result = highlight_rows(df_result, {foc["total_nw_change"]: "lightblue", foc["total_nw"]: "lightgreen"} )


    return df_result

def get_fin_overview_fig(fin_overview_table: Styler, foc, currency, invert = True) -> go.Figure:
    """
    Function to get the financial overview figure, based on the financial overview table
    params:
        fin_overview_table: Styler
            The financial overview table to be used for the figure

        foc: financial overview configuration dictionary

        currency: str
            The target currency, which represents the data of the fin_overview_table
            
        invert: bool
            If True, the signs of the following values will be inverted:
            - Income
            - Gains
            - Equity
            - Total Net Worth Change

        invert: bool
            If True, the sign of the values in the figure will be inverted
    """
    
    df_t = fin_overview_table.data.T
    df_t.index = df_t.index.astype(str)

    # Flip the sign of specific columns
    cols_to_change_sign = [
                           foc["income"]["name"],
                           foc["gains"]["name"],
                           foc["equity"]["name"],
                           foc["total_nw_change"]
                           ]
    
    if invert:
        df_t[cols_to_change_sign] = -df_t[cols_to_change_sign]

    # Define the rows for bar and line plots
    bar_rows = [foc["equity"]["name"],
                foc["expenses"]["name"],
                foc["income"]["name"],
                foc["gains"]["name"],
                foc["total_nw_change"]
                ]

    line_rows = [
                 foc["total_nw"]]

    # Create the figure
    fig = go.Figure()

    # Add bar traces (grouped bars)
    for col in bar_rows:
        fig.add_trace(go.Bar(
            x=df_t.index,
            y=df_t[col],
            name=col,
            yaxis='y1'
        ))

    # Add line traces on secondary y-axis
    for col in line_rows:
        fig.add_trace(go.Scatter(
            x=df_t.index,
            y=df_t[col],
            name=col,
            yaxis='y2',
            mode='lines+markers'
        ))

    # Update layout with dual y-axes
    fig.update_layout(
        title=f"Financial Overview in '{currency}'",
        xaxis=dict(
            title="Period",
            showgrid=True,           # Enables vertical gridlines
            gridcolor='lightgray',   # Optional: change gridline color
            gridwidth=1              # Optional: set gridline thickness
        ),
        yaxis=dict(
            title="P&L",
            showgrid=True,
            gridcolor='gray',
            gridwidth=1,
            zeroline=True
        ),
        yaxis2=dict(
            title="Net Worth",
            overlaying='y',
            side='right',
            showgrid=True,
            zeroline=True
            # range=[0, df_t[line_rows].max().max()*1.2]
        ),
        barmode='group',
        legend=dict(x=1.05, y=1),

        height=800,         # You can still fix the height
        autosize=True,      # Enable automatic sizing

        margin=dict(l=40, r=40, t=60, b=40)
    )

    return fig

def get_fin_overview_info(entries,
                           opts,
                           freq,
                           foc:dict,
                           *,
                           start_period = None,
                           qnt_periods = None,
                           end_period = None,
                           currency = None,
                           invert = True) -> tuple[Styler, go.Figure]:
    """
    Function to get the financial overview table, and figure, which displays it for the period from start_period to end_period

    params:
        entries: list of entries
        opts: dict of options
        freq: str
            pandas period alias (Y, Q, M, W, D)
                https://pandas.pydata.org/docs/user_guide/timeseries.html#timeseries-period-aliases
                
        foc: financial overview configuration dictionary

        start_period: Period, str, datetime, date or pandas.Timestamp  . pandas timeperiod representation
                    If not provided, the first date of the first entry will be used to determine the start_period

        qnt_periods: int
            number of periods to display

        end_period: Period, str, datetime, date or pandas.Timestamp  . pandas timeperiod representation
                    Used only if qnt_periods is not provided.
                    If not provided, the last date of the last entry will be used to determine the end_period
                    
        invert: bool
            If True, the signs of the following values will be inverted against the signed used in PTA
            - Income
            - Gains
            - Equity
            - Total Net Worth Change
            

    """
    
    foc_schema = Schema({
                "income":        {"name": str, "sql": str},
                "gains":         {"name": str, "sql": str},
                "expenses":      {"name": str, "sql": str},
                "equity":        {"name": str, "sql": str},
                "total_nw_change": str,
                "total_nw":        str,
                "unexplained_diff": str,
            })
    
    try:
        foc = foc_schema.validate(foc)
        
    except SchemaError as exc:
        
        raise ValueError(f"Invalid schema of the parameter 'foc': {foc}") from exc
    
    fin_overview_table = get_fin_overview_table(entries, opts, freq, foc, 
                                                start_period=start_period, 
                                                qnt_periods=qnt_periods, 
                                                end_period=end_period, 
                                                currency=currency)
    
    fin_overview_fig = get_fin_overview_fig(fin_overview_table, foc, currency, invert=invert)
    
    return fin_overview_table, fin_overview_fig



foc = {"income":{"name": "Income",
                            "sql": "account ~'^Income'  AND not account ~'^Income:Unrealized'"},

        "gains":{"name": "UnrealGains",
                            "sql": "account ~'^Income:Unrealized'"},

        "expenses":{"name": "Expenses",
                            "sql": "account ~'^Expenses'"},

        "equity":{"name": "Equity",
                            "sql": "account ~'^Equity'"},

        "total_nw_change": "Total Net  Worth Change",

        "total_nw": "Total Net Worth",

        "unexplained_diff": "Unexplained difference"}

In [15]:
fin_overview_table, fin_overview_fig = get_fin_overview_info(entries_eq, opts_eq, "Q", foc, currency = ANALYSIS_CURR)


In [16]:
fin_overview_table

,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,2017Q3
Expenses,"24,218.18","22,567.78","24,781.02","23,715.87","22,711.09","24,788.18","22,307.59","24,375.56","23,198.15","24,611.07","17,334.56"
Income,"-36,677.90","-31,438.20","-33,694.43","-32,570.57","-31,475.56","-36,227.16","-29,702.51","-32,569.35","-32,172.45","-36,779.43","-24,637.02"
UnrealGains,165.47,542.52,-822.58,-780.08,"-1,304.18","-2,413.56","1,338.72",-795.62,"-1,854.50","-5,716.71",817.08
Equity,"-3,490.52",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Total Net Worth Change,"-15,784.77","-8,327.90","-9,735.99","-9,634.78","-10,068.65","-13,852.54","-6,056.20","-8,989.41","-10,828.80","-17,885.07","-6,485.38"
Assets,"16,138.44","24,419.36","34,492.25","44,462.21","54,234.92","68,583.55","74,760.50","84,219.11","95,086.18","113,451.90","120,352.78"
Liabilities,-353.66,-306.68,-643.54,-978.70,-682.77,"-1,178.89","-1,299.66","-1,768.88","-1,807.13","-2,287.79","-2,703.29"
Total Net Worth,"15,784.78","24,112.68","33,848.71","43,483.51","53,552.15","67,404.66","73,460.84","82,450.23","93,279.05","111,164.11","117,649.49"
Unexplained difference,nan,-0.00,0.03,0.02,-0.00,-0.03,-0.02,-0.02,0.03,-0.01,-0.00


In [17]:
fin_overview_fig.show()

## <a id='toc1_7_'></a>[Exploring expenses](#toc0_)

In [18]:
#@title get_PL_acc_pivot
def get_PL_acc_pivot(entries,
                           opts,
                           freq,
                           q_str: str,
                           *,
                           combine_periods: bool = False,
                           start_period = None,
                           qnt_periods = None,
                           end_period = None,
                           currency: str | None= None,
                           max_row_levels: int =100) -> pd.DataFrame:
    
    """
    Function to get the P&L account pivot table for the period from start_period to end_period
    It is for the P&L account, because it sums the postings for the specific period, rather all posting from the 
    very beginning of the ledger up until the end date of the period, which would be the case for the Balance sheet 
    Accounts (e.g. Assets, Liabilities)
    
    q_str: str:
        The query string to filter the entries. It should be a valid SQL query string, which will be used to filter the
        entries. The query string should be in the format of a SQL WHERE clause, without the WHERE keyword.
        For example: "account ~ '^Expenses'"
        
    combine_periods: bool:
        If True, the periods, which form the columns of the pivot table will be combined into a single period with the 
        name 'all_periods'
        
    start_period: Period, str, datetime, date or pandas.Timestamp  . pandas timeperiod representation
        If not provided, the first date of the first entry will be used to determine the start_period
    qnt_periods: int
        number of periods to display. If provided, the end_period will be calculated as start_period + qnt_periods - 1
        
    end_period: Period, str, datetime, date or pandas.Timestamp  . pandas timeperiod representation
        Used only if qnt_periods is not provided (otherwise ignored).
        If not provided, the last date of the last entry will be used to determine the end_period
    
    currency: str | None:
        If the currency is provided, then the query will convert the position to the target currency, using the exchange 
        rate at the date of the transaction (which follows the standard accounting rules)

    """


    if not start_period:
        start_period = pd.Period(entries[0].date, freq)
    else:
        start_period = pd.Period(start_period, freq)

    if qnt_periods:
        end_period = start_period + qnt_periods - 1
    else:
        if not end_period:
            end_period = pd.Period(entries[-1].date, freq)
        else:
            end_period = pd.Period(end_period, freq)

    # if not currency:
    #     currency = opts["operating_currency"][0]

    # This is just to show types
    start_period: pd.Period = start_period
    end_period: pd.Period = end_period

    start_date_iso_str = start_period.start_time.strftime("%Y-%m-%d")
    end_date_iso_str = end_period.end_time.strftime("%Y-%m-%d")
    
    print(f"start_date_iso_str: {start_date_iso_str}")
    print(f"end_date_iso_str: {end_date_iso_str}")
    
    if currency:
    
        query = f"""
                select id, date, account, CONVERT(position,'{currency}',date) as amount, narration
                WHERE
                    ({q_str}) AND
                    date >= {start_date_iso_str} AND
                    date <= {end_date_iso_str}
                """
    else:
        query = f"""
                select id, date, account, position as amount, narration
                WHERE
                    ({q_str}) AND
                    date >= {start_date_iso_str} AND
                    date <= {end_date_iso_str}
                """
    df = beanquery2df(entries, opts,  query)
    
    if combine_periods:
        # Combine periods into a single period
        df['period'] = "all_periods"
        
    else:
        df['period'] = pd.PeriodIndex(df['date'], freq=freq)
    
    df_pivot = get_bean_pivot(df, column = "period", max_row_levels=max_row_levels, repeat_row_labels=False, 
                              swap_columns_groupping_order= False)
    
    # if currency:
    #     df_pivot = df_pivot.loc[:, pd.IndexSlice[['amount (USD)'], :]         ]
    
    # df_pivot = df_pivot[:f"amount ({currency})"]
    
    return df_pivot

In [19]:
expen_q_str = "account ~ '^Expenses'"
expenses_df_pivot = get_PL_acc_pivot(entries, options, "Q", expen_q_str, currency=ANALYSIS_CURR, combine_periods=False,
                                     max_row_levels=10)
# Here we only select columns, expressed in the Operating currency
expenses_df_pivot = expenses_df_pivot.loc[:, pd.IndexSlice[[f'amount ({ANALYSIS_CURR})'], :]         ]

start_date_iso_str: 1792-01-01
end_date_iso_str: 2017-09-30


In [20]:
expenses_df_pivot

amount (USD)          \
period                                                        2015Q1  2015Q2   
acc_L1    acc_L2      acc_L3        acc_L4   acc_L5                            
Financial Commissions _             _        _                  0.00    0.00   
          Fees        _             _        _                 12.00   12.00   
Food      Alcohol     _             _        _                  0.00    0.00   
          Coffee      _             _        _                  0.00    0.00   
          Groceries   _             _        _                471.81  686.54   
          Restaurant  _             _        _               1051.42 1130.89   
Health    Dental      Insurance     _        _                 20.30   17.40   
          Life        GroupTermLife _        _                170.24  145.92   
          Medical     Insurance     _        _                191.66  164.28   
          Vision      Insurance     _        _                296.10  253.80   
Home      Electricity _             _        _                195.00  195.00   
          Internet    _             _        _                240.31  240.17   
          Phone       _             _        _                183.94  208.58   
          Rent        _             _        _               7200.00 7200.00   
Taxes     Y2015       US            CityNYC  _               1224.44 1049.52   
                                    Federal  PreTax401k         0.00    0.00   
                                             _               7440.44 6377.52   
                                    Medicare _                746.34  639.72   
                                    SDI      _                  7.84    6.72   
                                    SocSec   _               1970.78 1689.24   
                                    State    _               2555.56 2190.48   
          Y2016       US            CityNYC  _                  0.00    0.00   
                                    Federal  PreTax401k         0.00    0.00   
                                             _                  0.00    0.00   
                                    Medicare _                  0.00    0.00   
                                    SDI      _                  0.00    0.00   
                                    SocSec   _                  0.00    0.00   
                                    State    _                  0.00    0.00   
          Y2017       US            CityNYC  _                  0.00    0.00   
                                    Federal  PreTax401k         0.00    0.00   
                                             _                  0.00    0.00   
                                    Medicare _                  0.00    0.00   
                                    SDI      _                  0.00    0.00   
                                    SocSec   _                  0.00    0.00   
                                    State    _                  0.00    0.00   
Transport Tram        _             _        _                240.00  360.00   
Vacation  _           _             _        _                  0.00    0.00   

                                                                         \
period                                                   2015Q3  2015Q4   
acc_L1    acc_L2      acc_L3        acc_L4   acc_L5                       
Financial Commissions _             _        _            35.80   35.80   
          Fees        _             _        _            12.00   12.00   
Food      Alcohol     _             _        _             0.00    0.00   
          Coffee      _             _        _            21.93    0.00   
          Groceries   _             _        _           528.59  367.06   
          Restaurant  _             _        _          1267.51 1088.92   
Health    Dental      Insurance     _        _            20.30   20.30   
          Life        GroupTermLife _        _           170.24  170.24   
          Medical     Insuranc

In [21]:
expenses_pivot_l1 = get_PL_acc_pivot(entries, options, "Q", expen_q_str, currency=ANALYSIS_CURR, combine_periods=False, max_row_levels=1)
expenses_pivot_l1 = expenses_pivot_l1.loc[:, pd.IndexSlice[[f'amount ({ANALYSIS_CURR})'], :]       ]
expenses_pivot_l1 


start_date_iso_str: 1792-01-01
end_date_iso_str: 2017-09-30


amount (USD)                                                        \
period          2015Q1   2015Q2   2015Q3   2015Q4   2016Q1   2016Q2   2016Q3   
acc_L1                                                                         
Financial        12.00    12.00    47.80    47.80    12.00    56.75    65.70   
Food           1523.23  1817.43  1818.03  1455.98  1428.19  1920.81  1545.55   
Health          678.30   581.40   678.30   678.30   581.40   678.30   581.40   
Home           7819.25  7843.75  7811.49  7829.93  7806.15  7826.92  7801.74   
Taxes         13945.40 11953.20 13945.40 13343.86 12523.35 13945.40 11953.20   
Transport       240.00   360.00   480.00   360.00   360.00   360.00   360.00   
Vacation          0.00     0.00     0.00     0.00     0.00     0.00     0.00   

                                              
period      2016Q4   2017Q1   2017Q2  2017Q3  
acc_L1                                        
Financial    20.95    56.75    12.00   38.85  
Food       1965.17  1493.84  1786.54 1406.04  
Health      678.30   581.40   678.30  484.50  
Home       7845.74  7833.27  7828.83 5204.17  
Taxes     13625.40 12872.89 13945.40 9961.00  
Transport   240.00   360.00   360.00  240.00  
Vacation      0.00     0.00     0.00    0.00

In [22]:
expenses_pivot_l1_USD = expenses_pivot_l1.loc[:, pd.IndexSlice[['amount (USD)'], :]         ]
expenses_pivot_l1_USD = expenses_pivot_l1_USD.sort_values(by=[('amount (USD)', '2015Q1')], ascending=False)
expenses_pivot_l1_USD


amount (USD)                                                        \
period          2015Q1   2015Q2   2015Q3   2015Q4   2016Q1   2016Q2   2016Q3   
acc_L1                                                                         
Taxes         13945.40 11953.20 13945.40 13343.86 12523.35 13945.40 11953.20   
Home           7819.25  7843.75  7811.49  7829.93  7806.15  7826.92  7801.74   
Food           1523.23  1817.43  1818.03  1455.98  1428.19  1920.81  1545.55   
Health          678.30   581.40   678.30   678.30   581.40   678.30   581.40   
Transport       240.00   360.00   480.00   360.00   360.00   360.00   360.00   
Financial        12.00    12.00    47.80    47.80    12.00    56.75    65.70   
Vacation          0.00     0.00     0.00     0.00     0.00     0.00     0.00   

                                              
period      2016Q4   2017Q1   2017Q2  2017Q3  
acc_L1                                        
Taxes     13625.40 12872.89 13945.40 9961.00  
Home       7845.74  7833.27  7828.83 5204.17  
Food       1965.17  1493.84  1786.54 1406.04  
Health      678.30   581.40   678.30  484.50  
Transport   240.00   360.00   360.00  240.00  
Financial    20.95    56.75    12.00   38.85  
Vacation      0.00     0.00     0.00    0.00

In [23]:
expenses_pivot_l1_USD = expenses_pivot_l1_USD.sort_values(by=('amount (USD)', '2015Q1'), ascending=False)
# expenses_pivot_l1_USD = expenses_pivot_l1_USD.loc[:, '2015Q1']
expenses_pivot_l1_USD = expenses_pivot_l1_USD.loc[:, 'amount (USD)']
expenses_pivot_l1_USD_total = add_total(expenses_pivot_l1_USD)
expenses_pivot_l1_USD_total 

period,2015Q1,2015Q2,2015Q3,2015Q4,2016Q1,2016Q2,2016Q3,2016Q4,2017Q1,2017Q2,2017Q3
Taxes,13945.40,11953.20,13945.40,13343.86,12523.35,13945.40,11953.20,13625.40,12872.89,13945.40,9961.00
Home,7819.25,7843.75,7811.49,7829.93,7806.15,7826.92,7801.74,7845.74,7833.27,7828.83,5204.17
Food,1523.23,1817.43,1818.03,1455.98,1428.19,1920.81,1545.55,1965.17,1493.84,1786.54,1406.04
Health,678.30,581.40,678.30,678.30,581.40,678.30,581.40,678.30,581.40,678.30,484.50
Transport,240.00,360.00,480.00,360.00,360.00,360.00,360.00,240.00,360.00,360.00,240.00
Financial,12.00,12.00,47.80,47.80,12.00,56.75,65.70,20.95,56.75,12.00,38.85
Vacation,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Total,24218.18,22567.78,24781.02,23715.87,22711.09,24788.18,22307.59,24375.56,23198.15,24611.07,17334.56


In [24]:
# @title stacked_area_from_wide function
def stacked_area_from_wide(df_wide, *, use_dates=False):
    """
    Parameters
    ----------
    df_wide : DataFrame
        Index      -> categories (Taxes, Home, …)
        Columns    -> periods  (2015Q1, 2015Q2, …  *either* strings or Periods)
    use_dates : bool, default False
        • False  → keep labels as strings (quick & simple)
        • True   → convert Periods to quarter‑end timestamps so the x‑axis
                   is a true date axis (nice for zooming, etc.)
    """
    # 1 . Transpose so periods become the index
    df_t = df_wide.T.copy()

    # 2 . Normalise the index
    if isinstance(df_t.index, pd.PeriodIndex):
        if use_dates:
            df_t.index = df_t.index.to_timestamp(how="end")   # 2015‑03‑31, etc.
        else:
            df_t.index = df_t.index.astype(str)               # "2015Q1", etc.
    else:
        # If the columns were ordinary strings ("2015Q1", …) we might still
        # want real dates when use_dates is True
        if use_dates:
            df_t.index = pd.PeriodIndex(df_t.index, freq="Q").to_timestamp("end")

    df_t.index.name = "period"        # make sure the name is known & lower‑case
    df_t = df_t.reset_index()         # 'period' becomes a normal column

    # 3 . Build the stacked‑area figure
    fig = px.area(
        df_t,
        x="period",
        y=df_wide.index,              # Taxes, Home, …
        title="Spending by category (stacked area)",
        labels={"value": "Amount (USD)", "period": "Period"},
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        yaxis_tickprefix="$",
        legend_title="Category",
        hovermode="x unified",
    )
    return fig

In [25]:
stacked_area_from_wide(expenses_pivot_l1_USD).show()

In [26]:
# @title get_groupped_bar_chart_fig function
def get_groupped_bar_chart_fig(df):
    """
    Generates a grouped bar chart Plotly figure from a DataFrame.

    Args:
        df (pd.DataFrame): DataFrame with categories in the index (level 'acc_L1')
                           and periods (years) as columns. Handles simple or
                           MultiIndex columns where the period is the last level.

    Returns:
        plotly.graph_objects.Figure: The generated Plotly figure object.
    """
    # Extract categories from the 'acc_L1' level of the index.
    categories = df.index.get_level_values('acc_L1').unique()
    # Get column names (periods/years)
    periods = df.columns

    # Create the figure object
    fig = go.Figure()

    # Add a bar trace for each period (column)
    for period in periods:
        # --- MODIFICATION START ---
        # Handle potential MultiIndex columns for the name property.
        # Assumes the relevant period identifier (like year) is the last element
        # if the column name is a tuple. Otherwise, uses the column name directly.
        if isinstance(period, tuple):
            # Convert the last element of the tuple to string for the name
            trace_name = str(period[-1])
        else:
            # If columns are not tuples, use them directly as strings
            trace_name = str(period)
        # --- MODIFICATION END ---

        # trace_name = str(period)

        fig.add_trace(go.Bar(
            x=categories, # x-axis represents the categories ('Taxes', 'Home', etc.)
            y=df[period], # y-axis represents the amount for that period and category
                          # Accessing column data works fine with tuples if columns are MultiIndex
            name=trace_name # Use the extracted/converted string name for the legend
        ))

    # Update the layout for a grouped bar chart appearance
    fig.update_layout(
        title_text='Expenses by Category and Period', # Chart title
        xaxis_title_text='Expense Category (acc_L1)', # x-axis label
        yaxis_title_text='Amount (USD)', # y-axis label
        barmode='group', # Key setting for grouped bars
        xaxis={'categoryorder':'array', 'categoryarray':categories}, # Maintain original category order
        legend_title_text='Period' # Title for the legend
    )

    # --- MODIFICATION: Return the figure instead of showing it ---
    return fig

In [27]:
fig = get_groupped_bar_chart_fig(expenses_pivot_l1_USD)

fig.show()

In [28]:
INP_FREQ = "Q" #@param ["Y", "Q", "M", "W", "D"]
INP_START_PERIOD = None #@param {type:"pd.Period"}
INP_QNT_PERIODS = None
INP_END_PERIOD = None #@param {type:"pd.Period"}	

expenses_pivot_for_sunburst = get_PL_acc_pivot(entries, options, INP_FREQ, expen_q_str, currency=ANALYSIS_CURR, combine_periods=True,
                                               start_period=INP_START_PERIOD,
                                               qnt_periods=INP_QNT_PERIODS,
                                               end_period=INP_END_PERIOD).loc[:, f'amount ({ANALYSIS_CURR})']
# expenses_pivot_for_sunburst

start_date_iso_str: 1792-01-01
end_date_iso_str: 2017-09-30


In [29]:
expenses_sunburst_fig =get_sunburst_figure_from_pivot(expenses_pivot_for_sunburst, column_to_pick='all_periods')
expenses_sunburst_fig.update_layout(margin=dict(t=30, l=0, r=0, b=0),
                    autosize=True,
                    width=700,
                    height=700,
                    title_text="Expenses",
                    # title_x=0.5,
                    # title_y=0.95,
                    title_font=dict(size=20),
                    font=dict(size=12)
                   )
expenses_sunburst_fig.show()

## <a id='toc1_8_'></a>[Filtering spesific transactions](#toc0_)

In [30]:
def check_if_to_filter(entry):
    file = StringIO("")

    printer.print_entry(entry, file=file)
    file.seek(0)
    entry_str = file.read()

    if "Income:US:ETrade:Gains" in entry_str:
        return True

filtered_entries = list(filter(check_if_to_filter, entries))

write_source = not IN_COLAB
display_entries(filtered_entries, write_source=write_source)

<evbeantools.printer_rich.display_entries.<locals>.DisplayableObject at 0x20a9afe2c30>

## <a id='toc1_9_'></a>[Prices information](#toc0_)

In [31]:
# Define the functions, needed to explore the price information

def draw_prices_circular_network(graph_data, radius=1):
    """Generates and displays a network graph using a circular layout with schema validation.

    This function first validates the input `graph_data` against a strict schema.
    If valid, it maps node identifiers to geometric positions on a circle and 
    renders the topology using Plotly.

    Args:
        graph_data (dict): A dictionary definition of the graph.
            Expected Schema:
            {
                <Node ID (str or int)>: {
                    "name": str,
                    "connections": [str or int],
                    Optional("special"): bool
                }
            }
        radius (int or float, optional): The geometric radius of the node circle. 
            Defaults to 1.

    Returns:
        None: Displays the interactive Plotly figure.
    
    Raises:
        SchemaError: If `graph_data` does not match the required structure.
    """
    
    # --- 1. Define Schema & Validate ---
    # We define the structure we expect. 
    # Or(str, int) allows keys/IDs to be either strings or integers.
    network_schema = Schema({
        Or(str, int): {
            "name": str,
            "connections": [Or(str, int)],   # Must be a list of IDs (strs or ints)
            Optional("special"): bool        # Key is optional; value must be bool
        }
    })

    try:
        # validate() returns the data if successful, or raises SchemaError
        validated_data = network_schema.validate(graph_data)
    except SchemaError as e:
        print(f"❌ Data Validation Error: {e}")
        return

    # --- 2. Setup Geometry ---
    node_ids = list(validated_data.keys())
    n_nodes = len(node_ids)
    
    # Create a mapping from Node ID -> Index (0 to N-1)
    id_to_index = {node_id: i for i, node_id in enumerate(node_ids)}
    
    # Calculate angles and coordinates
    angles = np.linspace(0, 2 * np.pi, n_nodes, endpoint=False)
    x_nodes = radius * np.cos(angles)
    y_nodes = radius * np.sin(angles)
    
    # --- 3. Build Edges ---
    edge_x = []
    edge_y = []
    
    for source_id, attributes in validated_data.items():
        source_idx = id_to_index[source_id]
        
        # We can safely iterate because schema validation ensured 'connections' is a list
        for target_id in attributes['connections']:
            if target_id in id_to_index:
                target_idx = id_to_index[target_id]
                edge_x.extend([x_nodes[source_idx], x_nodes[target_idx], None])
                edge_y.extend([y_nodes[source_idx], y_nodes[target_idx], None])

    # --- 4. Build Node Attributes ---
    node_labels = []
    node_colors = []
    
    COLOR_DEFAULT = 'navy'
    COLOR_SPECIAL = 'firebrick'

    for nid in node_ids:
        attr = validated_data[nid]
        node_labels.append(attr['name'])
        
        # Schema validation ensures 'special' is bool if present, but it might be missing
        # so we use .get() with a default.
        if attr.get('special', False):
            node_colors.append(COLOR_SPECIAL)
        else:
            node_colors.append(COLOR_DEFAULT)

    # --- 5. Create Plotly Traces ---
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1, color='#888'),
        hoverinfo='none',
        mode='lines'
    )

    node_trace = go.Scatter(
        x=x_nodes, y=y_nodes,
        mode='markers+text',
        text=node_labels,
        textposition="top center",
        hoverinfo='text',
        marker=dict(
            color=node_colors,
            size=25,
            line=dict(width=2, color='white')
        )
    )

    # --- 6. Render ---
    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        title='Currency Price Map Network',
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20,l=5,r=5,t=40),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="x", scaleratio=1),
                        plot_bgcolor='white'
                    ))
    
    fig.show()

def build_prices_graph_data(price_map, currencies=None, special_nodes=None):
    """Converts a Beancount PriceMap into a graph schema for visualization.

    This function extracts all currency pairs from the PriceMap to build a 
    network topology. It identifies every currency as a node and every 
    pricing relationship (base -> quote) as a connection.
    
    If there are currencies provided in the `currencies` argument which are not
    present in the PriceMap, they will still be included as isolated nodes.

    Args:
        price_map (dict): A Beancount PriceMap object (or similar dict) where 
            keys are tuples of strings: (base_currency, quote_currency).
            Values are ignored for the graph structure.
        
        currencies (list of str, optional): A list of currency codes used in 
            all entries. Defaults to None. This list will mostl of the time intersect with the currencies found in the 
            price_map, but it can also contain some additional currencies, which are not present in the price_map, 
            but are still used in transactions.
        
        special_nodes (list of str, optional): A list of currency codes 
            (e.g., ['USD', 'EUR']) that should be flagged as 'special'. On a visualization, these nodes will be 
            highlighted differently to indicate their importance or centrality in the output data. 
            These are typically operating currencies 
            Defaults to None. 

    Returns:
        dict: A dictionary conforming to the graph_data schema:
            {
                "USD": {"name": "USD", "connections": ["EUR"], "special": True},
                "EUR": {"name": "EUR", "connections": ["USD"], "special": False},
                "ISO": {"name": "ISO", "connections": [], "special": False},
                ...
            }
    """
    if currencies is None:
        currencies = []
    
    if special_nodes is None:
        special_nodes = []
    
    # Use a set for O(1) lookups
    special_set = set(special_nodes)
    
    graph_data = {}

    # 1. Pre-populate with known currencies (Nodes from Ledger)
    # This ensures isolated nodes appear even if they have no price history.
    for curr in currencies:
        graph_data[curr] = {
            "name": curr,
            "connections": [],
            "special": curr in special_set
        }

    # 2. Iterate over every currency pair in the price map (Edges)
    for base, quote in price_map.keys():
        
        # Ensure Base exists (might not be in the 'currencies' list)
        if base not in graph_data:
            graph_data[base] = {
                "name": base,
                "connections": [],
                "special": base in special_set
            }

        # Ensure Quote exists
        if quote not in graph_data:
            graph_data[quote] = {
                "name": quote,
                "connections": [],
                "special": quote in special_set
            }

        # Add the connection (Base -> Quote) if not already present
        if quote not in graph_data[base]["connections"]:
            graph_data[base]["connections"].append(quote)

    return graph_data

def plot_price_history(price_map: dict, pair: tuple):
    """Visualizes the historical exchange rates for a specific currency pair.

    Args:
        price_map (dict): The Beancount PriceMap object.
        pair (tuple): A tuple of two strings representing the currency pair 
                      to display, e.g., ('AAPL', 'USD').

    Returns:
        None: Displays the interactive Plotly figure.
    """
    
    # 1. Retrieve Data
    # The PriceMap values are lists of (date, number) tuples.
    # If the pair doesn't exist, we default to an empty list.
    history = price_map.get(pair, [])

    if not history:
        print(f"⚠️ No price history found for pair: {pair}")
        # Check if the inverse exists and suggest it
        inverse = (pair[1], pair[0])
        if inverse in price_map:
            print(f"   (However, the inverse pair {inverse} was found.)")
        return

    # 2. Unpack Data for Plotting
    # We separate dates and rates. We also ensure rates are converted to floats
    # because Beancount uses Decimals, which Plotly can sometimes struggle with.
    dates = [item[0] for item in history]
    rates = [float(item[1]) for item in history]

    # 3. Create Plotly Trace
    trace = go.Scatter(
        x=dates,
        y=rates,
        mode='lines+markers',
        name=f"{pair[0]}/{pair[1]}",
        line=dict(color='royalblue', width=2),
        marker=dict(size=4)
    )

    # 4. Configure Layout
    layout = go.Layout(
        title=f'Price History: {pair[0]} in {pair[1]}',
        xaxis=dict(
            title='Date',
            showgrid=True,
            gridcolor='#eee'
        ),
        yaxis=dict(
            title=f'Price ({pair[1]})',
            showgrid=True,
            gridcolor='#eee',
            zeroline=False
        ),
        plot_bgcolor='white',
        hovermode='x unified' # Shows the value when you hover anywhere on the x-axis
    )

    # 5. Render
    fig = go.Figure(data=[trace], layout=layout)
    fig.show()
    
def show_interactive_prices_diag(price_map):
    """Creates a Jupyter widget to explore price history for all pairs in the map.

    Args:
        price_map (dict): The Beancount PriceMap object.
    """
    
    # 1. extract and Sort Options
    # We create a list of tuples: [("Label to show", Actual Value), ...]
    # Example: [("EUR -> USD", ("EUR", "USD")), ("AAPL -> USD", ("AAPL", "USD"))]
    available_pairs = list(price_map.keys())
    
    if not available_pairs:
        print("⚠️ PriceMap is empty.")
        return

    # Sort alphabetically for better UX
    available_pairs.sort(key=lambda x: (x[0], x[1]))
    
    dropdown_options = [(f"{base} -> {quote}", (base, quote)) for base, quote in available_pairs]

    # 2. Create Widgets
    # Dropdown for selection
    pair_selector = widgets.Dropdown(
        options=dropdown_options,
        value=available_pairs[0], # Default to first pair
        description='Currency Pair:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='50%')
    )

    # Output area where the plot will be rendered
    plot_output = widgets.Output()

    # 3. Define Event Handler
    def on_pair_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            selected_pair = change['new']
            
            # Use the output widget context to capture the figure
            with plot_output:
                plot_output.clear_output(wait=True) # Clear previous chart
                plot_price_history(price_map, selected_pair)

    # Attach the handler
    pair_selector.observe(on_pair_change)

    # 4. Display Layout
    display(pair_selector, plot_output)

    # 5. Trigger Initial Draw
    # We manually trigger the draw for the default selected value
    with plot_output:
        plot_price_history(price_map, pair_selector.value)

    # --- Usage in Jupyter Notebook ---
    # interact_with_prices(price_map)
    
def get_posting_currencies(entries) -> set:
    """
    Scans entries and returns a set of all unique currencies used in Transaction postings.

    Args:
        entries (list): A list of Beancount directives (Transactions, Open, etc.).

    Returns:
        set: A set of unique currency strings (e.g., {'USD', 'EUR', 'AAPL'}).
    """
    currencies = set()
    
    for entry in entries:
        # We only care about entries that are Transactions
        if isinstance(entry, Transaction):
            for posting in entry.postings:
                # posting.units is an Amount(number, currency)
                currencies.add(posting.units.currency)
                
    return currencies



In [32]:
price_map = prices.build_price_map(entries)
posting_currencies = list(get_posting_currencies(entries))
prices_graph_data = build_prices_graph_data(price_map, posting_currencies, special_nodes=options["operating_currency"])
draw_prices_circular_network(prices_graph_data, radius=1)

In [33]:
show_interactive_prices_diag(price_map)

Dropdown(description='Currency Pair:', layout=Layout(width='50%'), options=(('GLD -> USD', ('GLD', 'USD')), ('…

Output()

In [34]:
show_interactive_fin_flow_diag(entries_eq, ["USD"])

In [35]:
from ipywidgets import Widget

def show_interactive_price_info(entries, options) -> Widget:
    """
    Displays (1) the circular price network and (2) an interactive price-history explorer
    together (no collapsing), in a single composite widget.
    """
    
    title_network = "Price Network"
    title_history = "Price History Explorer"
    radius = 1.0
    dropdown_width = "50%"
    
    opts = dict(options or {})

    # --- Build data ---
    price_map = prices.build_price_map(entries)
    posting_currencies = list(get_posting_currencies(entries))

    operating_currencies = options["operating_currency"]

    prices_graph_data = build_prices_graph_data(
        price_map,
        currencies=posting_currencies,
        special_nodes=operating_currencies,
    )
    

    # --- Output panes ---
    network_out = widgets.Output()
    history_out = widgets.Output()

    # --- Render network (top) ---
    with network_out:
        network_out.clear_output(wait=True)
        try:
            draw_prices_circular_network(prices_graph_data, radius=radius)
        except Exception as e:
            print(f"⚠️ Failed to render network graph: {e}")

    # --- Build history explorer (bottom) ---
    available_pairs = list(price_map.keys())
    available_pairs.sort(key=lambda x: (x[0], x[1]))

    if not available_pairs:
        pair_selector = widgets.Dropdown(
            options=[],
            description="Currency Pair:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width=dropdown_width),
            disabled=True,
        )
        plot_out = widgets.Output()
        with plot_out:
            print("⚠️ PriceMap is empty.")
    else:
        dropdown_options = [(f"{b} -> {q}", (b, q)) for b, q in available_pairs]

        pair_selector = widgets.Dropdown(
            options=dropdown_options,
            value=available_pairs[0],
            description="Currency Pair:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width=dropdown_width),
        )

        plot_out = widgets.Output()

        def redraw(pair):
            with plot_out:
                plot_out.clear_output(wait=True)
                plot_price_history(price_map, pair)

        def on_pair_change(change):
            if change.get("type") == "change" and change.get("name") == "value":
                redraw(change["new"])

        pair_selector.observe(on_pair_change, names="value")
        redraw(pair_selector.value)

    # --- Compose: both visible at once ---
    header_network = widgets.HTML(f"<h3 style='margin:0 0 8px 0;'>{title_network}</h3>")
    header_history = widgets.HTML(f"<h3 style='margin:16px 0 8px 0;'>{title_history}</h3>")

    root = widgets.VBox(
        [
            header_network,
            network_out,
            header_history,
            pair_selector,
            plot_out,
        ],
        layout=widgets.Layout(width="100%"),
    )

    # return root
    return root


# Example usage:
show_interactive_price_info(entries, options)



